# Monkey Patching in Ruby with `class_eval`

**The goal**: modify a class's behaviour at runtime — from a separate script — without touching the original source.

Three building blocks:

| Mechanic | What it opens |
|---|---|
| `SomeClass.class_eval { }` | instance method table of `SomeClass` |
| `alias_method :new_name, :old_name` | saves a copy before overwriting |
| `SomeClass.singleton_class.class_eval { }` | class-level (not instance) methods |

This notebook covers each mechanic in isolation, then assembles them into the **dry-run intercept pattern** used across all three demos.

## 1. `class_eval` — add a method to a class you don't own

In [1]:
String.class_eval do
  def shout
    upcase + '!!!'
  end
end

puts "hello".shout    # => HELLO!!!
puts "world".shout    # => WORLD!!!

HELLO!!!
WORLD!!!


## 2. `alias_method` — wrap an existing method

To intercept an existing method:
1. Save the original under a different name with `alias_method`
2. Define a replacement — the saved name lets you call through to the original

In [2]:
String.class_eval do
  alias_method :original_reverse, :reverse

  def reverse
    result = original_reverse
    puts "  [intercepted] reverse called on #{inspect} => #{result.inspect}"
    result
  end
end

"hello".reverse
"racecar".reverse

  [intercepted] reverse called on "hello" => "olleh"
  [intercepted] reverse called on "racecar" => "racecar"


"racecar"

## 3. `singleton_class.class_eval` — class-level methods

Class methods live on the *singleton class* of the class object itself.  
To patch them, open the singleton class instead of the class.

In [3]:
String.singleton_class.class_eval do
  alias_method :original_new, :new

  def new(*args, &block)
    puts "  [intercepted] String.new(#{args.inspect})"
    original_new(*args, &block)
  end
end

String.new("hello")
String.new

  [intercepted] String.new(["hello"])
  [intercepted] String.new([])


""

## 4. The dry-run pattern

The three mechanics compose into a single reusable shape:

```ruby
SomeClass.class_eval do
  alias_method :original_method, :method_name

  def method_name(*args)
    if DryRunContext.active?
      DryRunContext.capture(args)   # record what would have happened
      return                        # short-circuit -- side effect skipped
    end
    original_method(*args)          # real path -- side effect applied
  end
end
```

The context object is the only shared state between the patch and the caller.

In [4]:
module DryRunContext
  @active   = false
  @captures = []

  def self.enable!       = (@active = true; @captures = [])
  def self.disable!      = (@active = false)
  def self.active?       = @active
  def self.capture(data) = (@captures << data)
  def self.captures      = @captures.dup
end

Array.class_eval do
  alias_method :original_push, :push

  def push(*items)
    if DryRunContext.active?
      DryRunContext.capture({ would_add: items, array_before: dup })
      return self   # no-op: don't actually push
    end
    original_push(*items)
  end
end

arr = [1, 2, 3]

# Phase 1 -- dry run
DryRunContext.enable!
arr.push(99)
puts "Dry run  -- arr: #{arr.inspect}"        # [1, 2, 3]     unchanged
puts "Captured:        #{DryRunContext.captures}"

# Phase 2 -- real run
DryRunContext.disable!
arr.push(99)
puts "Real run -- arr: #{arr.inspect}"        # [1, 2, 3, 99]

Dry run  -- arr: [1, 2, 3]
Captured:        [{:would_add=>[99], :array_before=>[1, 2, 3]}]
Real run -- arr: [1, 2, 3, 99]


## Up next

The following notebooks apply this exact pattern to real frameworks:

| Notebook | Patched class | Intercepted method | Rollback? |
|---|---|---|---|
| `01_active_record` | `ActiveRecord::Base` | `before_save` callback | yes |
| `02_graphql` | `AppSchema` (singleton class) | `execute` class method | capture only |
| `03_karafka` | `WaterDrop::Producer` | `produce_sync` | capture only |
| `04_all_together` | all three | all three, same process | AR only |